1. What is repartition() and why to use it 
2. Does repartition() cause a shuffle and why?
3. Can repartition() increase and decrease the no of partition
4. diffrence between df.repartition(4) and df.repartition(4,"id")
5. What happend if you repartiton 10GB of data into 10,000 partition 
6. If one key has million of records will repartition(10,"customer_id") always balance the data
7. why hash partitioning create skew
8. In what situtaion would you use repartiton() 

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataSkewBroadcastJoinDemo")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "true")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark Version       :", spark.version)
print("Application Name    :", sc.appName)
print("Master              :", sc.master)
print("Default Parallelism :", sc.defaultParallelism)
print("Shuffle Partitions  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI            :", sc.uiWebUrl)

In [ ]:
df=spark.range(0,100)
print(df.rdd.getNumPartitions())

In [ ]:
df2=df.repartition(3)
print(df2.rdd.getNumPartitions())

In [ ]:
df3=df2.repartition(5)
print(df3.rdd.getNumPartitions())

In [ ]:
data=[
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David"),
    (5, "Eve"),
    (6, "Frank"),
    (7, "Grace"),
    (8, "Henry"),
    (9, "Ivy"),
    (10, "Jack")
]

df=spark.createDataFrame(data, ["id", "name"])
df.show()

In [ ]:
df.rdd.getNumPartitions()

In [ ]:
from pyspark.sql import functions as F
df_with_partition = df.withColumn("partition_id", F.spark_partition_id())
df_with_partition.show()

In [ ]:
df_rep=df.repartition(3)

In [ ]:
df_rep.rdd.getNumPartitions()

In [ ]:
df_rep_with_partition = df_rep.withColumn("partition_id", F.spark_partition_id())
df_rep_with_partition.show()

In [ ]:
df_rep_new=df.repartition(10)
df_rep_new.rdd.getNumPartitions()


In [ ]:
df_rep_with_partition =df_rep_new.withColumn("partition_id", F.spark_partition_id())
df_rep_with_partition.show()

In [ ]:
df_rep_new.explain()

In [ ]:
from pyspark.sql.functions import spark_partition_id
df=spark.range(1,21)
df.show()

In [ ]:
df_check=df.withColumn("partition_id", spark_partition_id())
df_check.show()

In [ ]:
df_check.withColumn("partition_id", spark_partition_id()).groupBy("partition_id").count().orderBy("partition_id").show()

In [ ]:
from pyspark.sql.functions import spark_partition_id
df=spark.range(1,21)
df_by_id=df.repartition(5, "id")

df_by_id.withColumn("partition_id",spark_partition_id()).orderBy("id").show(50,False)

In [ ]:
df_by_id.explain()

In [ ]:
from pyspark.sql.functions import spark_partition_id
data=[
    (1,"A"),
    (1,"B"),
    (2,"C"),
    (2,"D"),
    (3,"E"),
    (4,"F"),
    (4,"G"),
]

df=spark.createDataFrame(data, ["id", "name"])
df.show()

In [ ]:
df_hash=df.repartition(5, "id")


In [ ]:
df_hash.withColumn("partition_id",spark_partition_id()).orderBy("id").show(50,False)

In [ ]:
from pyspark.sql.functions import hash,col

df=spark.createDataFrame(
    [(1,),(2,),(3,),(4,),(5,)]
)

df.withColumn("hash_value", hash(col("_1"))).show()

In [ ]:
from pyspark.sql.functions import hash,col,pmod,lit 

df2=(
    df
    .withColumn("hash_value", hash(col("_1")))
    .withColumn(
        "expected_partition",
    pmod(col("hash_value"), lit(5))
    )
)

df2.show()

In [ ]:
df.printSchema()

In [ ]:
df=spark.range(0,1_000_000)

df.rdd.getNumPartitions()

In [ ]:
df_small_partitions=df.repartition(2)
print(df_small_partitions.rdd.getNumPartitions())

In [ ]:
(df_small_partitions
.withColumn("partition_id",spark_partition_id())
.groupBy("partition_id")
.count()
.orderBy("partition_id")
.show()
)

In [ ]:
df_better=df_small_partitions.repartition(10)
print(df_better.rdd.getNumPartitions())

In [ ]:
(df_better
.withColumn("partition_id",spark_partition_id())
.groupBy("partition_id")
.count()
.orderBy("partition_id")
.show()
)

# Colasec 

In [ ]:
from pyspark.sql.functions import spark_partition_id
df=spark.range(1,21)
print("Initial partitions: ", df.rdd.getNumPartitions())

In [ ]:
df_coalesced=df.coalesce(2)
print("After coalesce: ", df_coalesced.rdd.getNumPartitions())

In [ ]:
(
    df_coalesced
    .withColumn("partition_id",spark_partition_id())
    .orderBy("partition_id","id")
    .show(50,False)
)

In [ ]:
df_coalesced.explain()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import spark_partition_id,col
spark=(
    SparkSession.builder
    .appName("CoalesceDemo")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext

print("Spark Version       :", spark.version)
print("Application Name    :", sc.appName)
print("Master              :", sc.master)
print("Default Parallelism :", sc.defaultParallelism)
print("Shuffle Partitions  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI            :", sc.uiWebUrl)

In [ ]:
df=spark.range(
    start=1,
    end=1_000_001,
    numPartitions=8
)

In [ ]:
print("Initial partitions: ", df.rdd.getNumPartitions())

In [ ]:
def count_partition_rows(partition_id,rows):
    count=0
    for row in rows:
        count+=1
    yield (partition_id,count)

partition_counts=(
    df.rdd.mapPartitionsWithIndex(count_partition_rows)
    .collect()
)

partition_counts

In [ ]:
filtered_df=df.filter(
    col("id") % 20 == 0
)

In [ ]:
filtered_df.count()

In [ ]:
print(
    "Partition after filter:", filtered_df.rdd.getNumPartitions()
)

In [ ]:
def count_partition_rows(partition_id,rows):
    count=0
    for row in rows:
        count+=1
    yield (partition_id,count)

partition_counts=(
    filtered_df.rdd.mapPartitionsWithIndex(count_partition_rows)
    .collect()
)

partition_counts

In [ ]:
filtered_df.write.mode("overwrite").parquet("output/coalesced_output/normal_output/")

In [ ]:
filtered_df.coalesce(2).write.mode("overwrite").parquet("output/coalesced_output/normal_output/")

# Data Skew Practical 

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import spark_partition_id,col
spark=(
    SparkSession.builder
    .appName("DataSkewDemo")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext

print("Spark Version       :", spark.version)
print("Application Name    :", sc.appName)
print("Master              :", sc.master)
print("Default Parallelism :", sc.defaultParallelism)
print("Shuffle Partitions  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI            :", sc.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/06 18:59:41 WARN Utils: Your hostname, Anujs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.124 instead (on interface en0)
26/10/06 18:59:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/anujshahdeo/Documents/Full_Stack_AWS_Data_Engineering/full-stack-aws-data-engineering/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/06 18:59:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin

Spark Version       : 4.2.0
Application Name    : DataSkewDemo
Master              : local[*]
Default Parallelism : 8
Shuffle Partitions  : 200
Spark UI            : http://192.168.29.124:4040


In [ ]:
data=(
    [("A",)]*90000 +
    [("B",)]*5000 +
    [("C",)]*3000 +
    [("D",)]*2000 
)

df=spark.createDataFrame(data, ["key"])

In [ ]:
df.count()

In [ ]:
df.groupBy("key").count().orderBy("count", ascending=False).show()


In [ ]:
df_skew=df.repartition(4, "key")

In [ ]:
from pyspark.sql.functions import spark_partition_id,col
(
    df_skew
    .withColumn("partition_id",spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")    
    .show()
)

In [ ]:
from pyspark.sql.functions import spark_partition_id,col
(
    df_skew
    .withColumn("partition_id",spark_partition_id())
    .groupBy("key", "partition_id")
    .count()
    .orderBy("partition_id","key")
    .show()
)

In [ ]:
def count_rows_per_partition(index,iterator):
    count=sum(1 for _ in iterator)
    yield (index,count)

df_skew.rdd.mapPartitionsWithIndex(count_rows_per_partition).collect()

# Solution Skew - Salting 

In [ ]:
data=(
    [("A",)]*90000 +
    [("B",)]*5000 +
    [("C",)]*3000 +
    [("D",)]*2000 
)

df=spark.createDataFrame(data, ["key"])

In [ ]:
from pyspark.sql.functions import floor, rand, concat, col, lit
salted_df=(
    df
    .withColumn(
        "salt",
        floor(rand()*4)
    )
    .withColumn(
        "salted_key",
        concat(col("key"),lit("_"),col("salt"))
    )
)

In [ ]:
salted_df.show(20,False)

In [ ]:
(
    salted_df
    .filter(col("key")=="A")
    .groupBy("salted_key")
    .count()
    .orderBy("salted_key")
    .show()
)

In [ ]:
salted_rep=salted_df.repartition(4, "salted_key")

In [ ]:
from pyspark.sql.functions import spark_partition_id,col
(
    salted_rep
    .withColumn("partition_id",spark_partition_id())
    .groupBy("salted_key","partition_id")
    .count()
    .orderBy("partition_id","salted_key")    
    .show()
)

In [5]:
from pyspark.sql import functions as F
df=(
    spark.range(0,100000)
    .withColumn(
        "customer_id",
        F.when(F.col("id") < 80000, F.lit(101))
        .when(F.col("id") < 85000, F.lit(102))
        .when(F.col("id") < 90000, F.lit(103))
        .when(F.col("id") <95000, F.lit(104))
        .otherwise(F.lit(105))
    )

    .withColumn(
        "amount",
        F.when(F.col("customer_id") == 101, 100)
        .when(F.col("customer_id") == 102, 200)
        .when(F.col("customer_id") == 103, 300)
        .when(F.col("customer_id") == 104, 400)
        .otherwise(F.lit(500))
    )
    .select("customer_id","amount")
)

In [6]:
df.count()

100000

In [7]:
df.show()

+-----------+------+
|customer_id|amount|
+-----------+------+
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
|        101|   100|
+-----------+------+
only showing top 20 rows


In [8]:
df.groupBy("customer_id").count().orderBy("count", ascending=False).show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|        101|80000|
|        103| 5000|
|        102| 5000|
|        105| 5000|
|        104| 5000|
+-----------+-----+



In [9]:
df_skew=df.repartition(4, "customer_id")

In [10]:
print("Number of partitions in df_skew: ", df_skew.rdd.getNumPartitions())

Number of partitions in df_skew:  4


In [11]:
def count_rows_per_partition(index,iterator):
    count=sum(1 for _ in iterator)
    yield (index,count)

partition_count=(
    df_skew.rdd.mapPartitionsWithIndex(count_rows_per_partition)
    .collect()
)
partition_count

[(0, 80000), (1, 5000), (2, 10000), (3, 5000)]

In [12]:
from pyspark.sql.functions import spark_partition_id,col
(
    df_skew
    .withColumn("partition_id",spark_partition_id())
    .groupBy("customer_id", "partition_id")
    .count()
    .orderBy("partition_id","customer_id")
    .show()
)

+-----------+------------+-----+
|customer_id|partition_id|count|
+-----------+------------+-----+
|        101|           0|80000|
|        105|           1| 5000|
|        102|           2| 5000|
|        104|           2| 5000|
|        103|           3| 5000|
+-----------+------------+-----+



In [27]:
from pyspark.sql import functions as F
salted_df=(
    df
    .withColumn(
        "salt",
        F.when(
            F.col("customer_id") == 101, 
            F.floor(F.rand(seed=42)*15)
        ).otherwise(F.lit(0))
        )
        .withColumn(
            "salted_customer_id",
            F.concat(F.col("customer_id").cast("string"),
                     F.lit("_"),
                     F.col("salt").cast("string")) 
    )
    .cache()
)


In [28]:
salted_df.count()

100000

In [22]:
salted_df\
.select("customer_id","salt","salted_customer_id")\
.show(20,False)

+-----------+----+------------------+
|customer_id|salt|salted_customer_id|
+-----------+----+------------------+
|101        |6   |101_6             |
|101        |5   |101_5             |
|101        |8   |101_8             |
|101        |2   |101_2             |
|101        |6   |101_6             |
|101        |5   |101_5             |
|101        |9   |101_9             |
|101        |0   |101_0             |
|101        |9   |101_9             |
|101        |7   |101_7             |
|101        |4   |101_4             |
|101        |6   |101_6             |
|101        |3   |101_3             |
|101        |8   |101_8             |
|101        |7   |101_7             |
|101        |7   |101_7             |
|101        |0   |101_0             |
|101        |6   |101_6             |
|101        |0   |101_0             |
|101        |8   |101_8             |
+-----------+----+------------------+
only showing top 20 rows


In [29]:
(
    salted_df
    .filter(F.col("customer_id") == 101)
    .groupBy("salted_customer_id")
    .count()
    .orderBy("salted_customer_id")
    .show()
)

+------------------+-----+
|salted_customer_id|count|
+------------------+-----+
|             101_0| 5359|
|             101_1| 5353|
|            101_10| 5282|
|            101_11| 5325|
|            101_12| 5295|
|            101_13| 5365|
|            101_14| 5225|
|             101_2| 5392|
|             101_3| 5175|
|             101_4| 5304|
|             101_5| 5449|
|             101_6| 5375|
|             101_7| 5419|
|             101_8| 5412|
|             101_9| 5270|
+------------------+-----+



In [33]:
salted_rep=salted_df.repartition(12, "salted_customer_id")

In [35]:
print("Number of partitions in df_skew: ", df_skew.rdd.getNumPartitions())

Number of partitions in df_skew:  4


In [36]:
def count_rows_per_partition(index,iterator):
    count=sum(1 for _ in iterator)
    yield (index,count)

salted_partition_count=(
    salted_rep.rdd.mapPartitionsWithIndex(count_rows_per_partition)
    .collect()
)
salted_partition_count

[(0, 10225),
 (1, 5000),
 (2, 0),
 (3, 10419),
 (4, 31747),
 (5, 16084),
 (6, 5304),
 (7, 5392),
 (8, 0),
 (9, 0),
 (10, 0),
 (11, 15829)]

# Solution - Broadcast

transection= 500 GB 
customer_type= 20 MB 

--> customer_id 

In [37]:
from pyspark.sql import functions as F
transections=(
    spark.range(0,100000)
    .withColumn(
        "customer_id",
        F.when(F.col("id") < 80000, F.lit(101))
        .when(F.col("id") < 85000, F.lit(102))
        .when(F.col("id") < 90000, F.lit(103))
        .when(F.col("id") <95000, F.lit(104))
        .otherwise(F.lit(105))
    )
    .withColumn(
        "amount",F.lit(100))
)

In [38]:
customer_type=spark.createDataFrame(
    [
        (101, "Premium"),
        (102, "Regular"),
        (103, "Regular"),
        (104, "Gold"),
        (105, "Silver")
    ], 
    ["customer_id", "customer_type"]
)



In [39]:
normal_join=transections.join(
    customer_type, 
    "customer_id", 
    "left"
)
normal_join.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#1363, id#1362L, 100 AS amount#1364, customer_type#1366]
   +- SortMergeJoin [cast(customer_id#1363 as bigint)], [customer_id#1365L], LeftOuter
      :- Sort [cast(customer_id#1363 as bigint) ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(cast(customer_id#1363 as bigint), 200), ENSURE_REQUIREMENTS, [plan_id=774]
      :     +- Project [id#1362L, CASE WHEN (id#1362L < 80000) THEN 101 WHEN (id#1362L < 85000) THEN 102 WHEN (id#1362L < 90000) THEN 103 WHEN (id#1362L < 95000) THEN 104 ELSE 105 END AS customer_id#1363]
      :        +- Range (0, 100000, step=1, splits=8)
      +- Sort [customer_id#1365L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(customer_id#1365L, 200), ENSURE_REQUIREMENTS, [plan_id=775]
            +- Filter isnotnull(customer_id#1365L)
               +- Scan ExistingRDD[customer_id#1365L,customer_type#1366]




In [40]:
broadcast_join=transections.join(
    F.broadcast(customer_type),
    "customer_id",
    "left"
)
broadcast_join.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#1363, id#1362L, 100 AS amount#1364, customer_type#1366]
   +- BroadcastHashJoin [cast(customer_id#1363 as bigint)], [customer_id#1365L], LeftOuter, BuildRight, false, false
      :- Project [id#1362L, CASE WHEN (id#1362L < 80000) THEN 101 WHEN (id#1362L < 85000) THEN 102 WHEN (id#1362L < 90000) THEN 103 WHEN (id#1362L < 95000) THEN 104 ELSE 105 END AS customer_id#1363]
      :  +- Range (0, 100000, step=1, splits=8)
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=804]
         +- Filter isnotnull(customer_id#1365L)
            +- Scan ExistingRDD[customer_id#1365L,customer_type#1366]


